In [0]:
%sql
create or replace table b_sql.b_practice.customer_orders (
order_id integer,
customer_id integer,
order_date date,
order_amount integer
);

insert into b_sql.b_practice.customer_orders values(1,100,cast('2022-01-01' as date),2000),(2,200,cast('2022-01-01' as date),2500),(3,300,cast('2022-01-01' as date),2100)
,(4,100,cast('2022-01-02' as date),2000),(5,400,cast('2022-01-02' as date),2200),(6,500,cast('2022-01-02' as date),2700)
,(7,100,cast('2022-01-03' as date),3000),(8,400,cast('2022-01-03' as date),1000),(9,600,cast('2022-01-03' as date),3000)
;
select * from b_sql.b_practice.customer_orders;

In [0]:
%sql
with cte as (select  *, min(order_date) over(partition by customer_id order by order_date) as min_date, case when order_date= min(order_date) over(partition by customer_id order by order_date) then 1 else 0 end as flag  from b_sql.b_practice.customer_orders order by customer_id)
select order_date,sum(flag) as new, count(*)-sum(flag) as old from cte group by order_date

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
if spark.catalog.tableExists("b_sql.b_practice.customer_orders"):
    df_order=spark.read.table("b_sql.b_practice.customer_orders")
    df_order.display()
else:
    print("table not exist")

In [0]:
min_date=Window.partitionBy(col("customer_id")).orderBy(col("order_date"))
df_order=df_order.withColumn("min_order_date",min(col("order_date")).over(min_date)).orderBy("customer_id")
df_flag=df_order.withColumn("flag",when(col("order_date")==col("min_order_date"),1).otherwise(0))
df_final=df_flag.groupBy("order_date").agg(count("*").alias("total_order"),sum("flag").alias("new"),(count("*")-sum("flag")).alias("old"))
df_final.display()